![A car dashboard with lots of new technical features.](images/dashboard.jpg)

You're working for a well-known car manufacturer who is looking at implementing LLMs into vehicles to provide guidance to drivers. You've been asked to experiment with integrating car manuals with an LLM to create a context-aware chatbot.

**Note: API calls are routed through OpenRouter. Put your key in a `.env` file (see `.env.example`) - never hardcode API keys.**

This notebook is a thin demo: all logic lives in the `car_rag` package (`src/car_rag/`). For scripted use, prefer the CLI (`car-rag ingest / ask / eval`) - see `README.md`.

In [ ]:
import sys
sys.path.insert(0, "src")

from dotenv import load_dotenv
load_dotenv()  # reads .env in the project root

from car_rag import rag
from car_rag.config import get_settings
from car_rag.rerank import OpenRouterReranker

settings = get_settings()
settings.require_api_key()
print("chat:", settings.openrouter_chat_model, "| judge:", settings.eval_llm_model)

In [ ]:
# Ingest: load the manual HTML, chunk it, build the Chroma index
docs = rag.load_documents(settings.data_path)
chunks = rag.split_documents(docs, settings.chunk_size, settings.chunk_overlap)
vectorstore = rag.build_vectorstore(
    chunks, rag.make_embeddings(settings), settings.chroma_dir
)
print(f"Indexed {len(chunks)} chunks.")

In [ ]:
# Ask: dense retrieval
llm = rag.make_chat_llm(settings)
store = rag.load_vectorstore(rag.make_embeddings(settings), settings.chroma_dir)
dense = rag.build_retriever("dense", store, None, settings.retriever_k)
question = (
    "The Gasoline Particular Filter Full warning has appeared. "
    "What does this mean and what should I do about it?"
)
print(rag.answer(question, dense, llm)["response"])

In [ ]:
# Ask: hybrid retrieval (BM25 + vector) with OpenRouter reranking
reranker = OpenRouterReranker(
    api_key=settings.require_api_key(),
    base_url=settings.openrouter_base_url,
    model=settings.rerank_model,
    top_n=settings.rerank_top_n,
)
hybrid = rag.build_retriever(
    "hybrid", store, chunks, settings.retriever_k, settings.hybrid_bm25_weight
)
print(rag.answer(question, hybrid, llm, reranker)["response"])

## Next steps

- `car-rag ask "..." --retriever hybrid --rerank` - ask from the terminal
- `python evals/run_evals.py --dataset evals/datasets/golden.jsonl --retriever hybrid --rerank` - score with RAGAS
- `python evals/compare_reports.py evals/reports/<a>.json evals/reports/<b>.json` - compare runs
- Metrics: Output Accuracy (`AnswerAccuracy`), Faithfulness, Context Precision (`LLMContextPrecisionWithReference`)